<a href="https://colab.research.google.com/github/Bloop15/Machine-Learning--II/blob/main/FOIL_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],
    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],
    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],
    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],
    'PlayTennis': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Dataset shape:", df.shape)

df

Dataset created successfully!
Dataset shape: (14, 5)


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
X = df.drop("PlayTennis", axis=1)
y = df["PlayTennis"]

print("Input attributes:")
print(X.columns.tolist())

print("\nTarget classes:")
print(y.value_counts())

print("\nNumber of positive examples:", (y == "Yes").sum())
print("Number of negative examples:", (y == "No").sum())

Input attributes:
['Outlook', 'Temperature', 'Humidity', 'Wind']

Target classes:
PlayTennis
Yes    9
No     5
Name: count, dtype: int64

Number of positive examples: 9
Number of negative examples: 5


In [4]:
examples = []

for i, row in df.iterrows():
    example = {
        "Outlook": row["Outlook"],
        "Temperature": row["Temperature"],
        "Humidity": row["Humidity"],
        "Wind": row["Wind"],
        "PlayTennis": row["PlayTennis"]
    }
    examples.append(example)

positive_examples = [
    ex for ex in examples if ex["PlayTennis"] == "Yes"
]

negative_examples = [
    ex for ex in examples if ex["PlayTennis"] == "No"
]

print("Total examples:", len(examples))
print("Positive examples:", len(positive_examples))
print("Negative examples:", len(negative_examples))

print("\nFirst-order representation of the first example:")
print(positive_examples[0])

Total examples: 14
Positive examples: 9
Negative examples: 5

First-order representation of the first example:
{'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'Yes'}


In [5]:
import math

def foil_gain(p0, n0, p1, n1):
    if p1 == 0 or (p0 + n0) == 0 or (p1 + n1) == 0:
        return 0

    return p1 * (
        math.log2(p1 / (p1 + n1))
        - math.log2(p0 / (p0 + n0))
    )

print("FOIL Gain function defined successfully!")

FOIL Gain function defined successfully!


In [6]:
attributes = X.columns.tolist()

candidate_literals = []

for attribute in attributes:
    for value in X[attribute].unique():
        candidate_literals.append((attribute, value))

print("Total candidate literals:", len(candidate_literals))

print("\nCandidate literals:")
for attribute, value in candidate_literals:
    print(f"{attribute}(X, {value})")

Total candidate literals: 10

Candidate literals:
Outlook(X, Sunny)
Outlook(X, Overcast)
Outlook(X, Rain)
Temperature(X, Hot)
Temperature(X, Mild)
Temperature(X, Cool)
Humidity(X, High)
Humidity(X, Normal)
Wind(X, Weak)
Wind(X, Strong)


In [7]:
p0 = len(positive_examples)
n0 = len(negative_examples)

literal_gains = []

for attribute, value in candidate_literals:

    matching = [
        ex for ex in examples
        if ex[attribute] == value
    ]

    p1 = sum(ex["PlayTennis"] == "Yes" for ex in matching)
    n1 = sum(ex["PlayTennis"] == "No" for ex in matching)

    gain = foil_gain(p0, n0, p1, n1)

    literal_gains.append({
        "Literal": f"{attribute}(X, {value})",
        "Positive": p1,
        "Negative": n1,
        "FOIL Gain": round(gain, 4)
    })

gain_df = pd.DataFrame(literal_gains)

gain_df = gain_df.sort_values(
    by="FOIL Gain",
    ascending=False
)

print("FOIL Gain for Candidate Literals:")
display(gain_df)

FOIL Gain for Candidate Literals:


,Literal,Positive,Negative,FOIL Gain
1,"Outlook(X, Overcast)",4,0,2.5497
7,"Humidity(X, Normal)",6,1,2.4902
8,"Wind(X, Weak)",6,2,1.3344
5,"Temperature(X, Cool)",3,1,0.6672
4,"Temperature(X, Mild)",4,2,0.2099
2,"Outlook(X, Rain)",3,2,-0.2986
3,"Temperature(X, Hot)",2,2,-0.7251
9,"Wind(X, Strong)",3,3,-1.0877
0,"Outlook(X, Sunny)",2,3,-1.3690
6,"Humidity(X, High)",3,4,-1.7549


In [8]:
def learn_foil_rule(positive_examples, negative_examples):

    rule = []
    remaining_positive = positive_examples.copy()
    remaining_negative = negative_examples.copy()

    while remaining_negative:

        best_literal = None
        best_gain = float("-inf")

        p0 = len(remaining_positive)
        n0 = len(remaining_negative)

        for attribute, value in candidate_literals:

            p1 = sum(
                ex[attribute] == value
                for ex in remaining_positive
            )

            n1 = sum(
                ex[attribute] == value
                for ex in remaining_negative
            )

            gain = foil_gain(p0, n0, p1, n1)

            if gain > best_gain:
                best_gain = gain
                best_literal = (attribute, value)

        if best_literal is None or best_gain <= 0:
            break

        rule.append(best_literal)

        remaining_positive = [
            ex for ex in remaining_positive
            if ex[best_literal[0]] == best_literal[1]
        ]

        remaining_negative = [
            ex for ex in remaining_negative
            if ex[best_literal[0]] == best_literal[1]
        ]

    return rule


print("FOIL rule learning function defined successfully!")

FOIL rule learning function defined successfully!


In [9]:
first_rule = learn_foil_rule(
    positive_examples,
    negative_examples
)

print("Learned FOIL Rule:")

if first_rule:
    conditions = [
        f"{attribute}(X, {value})"
        for attribute, value in first_rule
    ]

    print("IF", " AND ".join(conditions))
    print("THEN PlayTennis(X, Yes)")
else:
    print("No rule could be learned.")

Learned FOIL Rule:
IF Outlook(X, Overcast)
THEN PlayTennis(X, Yes)


In [10]:
def rule_covers(example, rule):
    return all(
        example[attribute] == value
        for attribute, value in rule
    )


remaining_positive = positive_examples.copy()
foil_rules = []

while remaining_positive:

    rule = learn_foil_rule(
        remaining_positive,
        negative_examples
    )

    if not rule:
        break

    covered = [
        ex for ex in remaining_positive
        if rule_covers(ex, rule)
    ]

    if not covered:
        break

    foil_rules.append(rule)

    remaining_positive = [
        ex for ex in remaining_positive
        if not rule_covers(ex, rule)
    ]


print("Complete FOIL Rule Set:")
print("=" * 45)

for i, rule in enumerate(foil_rules, 1):

    conditions = [
        f"{attribute}(X, {value})"
        for attribute, value in rule
    ]

    print(f"\nRule {i}:")
    print("IF", " AND ".join(conditions))
    print("THEN PlayTennis(X, Yes)")

print("\nTotal rules learned:", len(foil_rules))
print("Remaining positive examples:", len(remaining_positive))

Complete FOIL Rule Set:

Rule 1:
IF Outlook(X, Overcast)
THEN PlayTennis(X, Yes)

Rule 2:
IF Humidity(X, Normal) AND Wind(X, Weak)
THEN PlayTennis(X, Yes)

Rule 3:
IF Temperature(X, Mild) AND Humidity(X, Normal)
THEN PlayTennis(X, Yes)

Rule 4:
IF Outlook(X, Rain) AND Wind(X, Weak)
THEN PlayTennis(X, Yes)

Total rules learned: 4
Remaining positive examples: 0


In [11]:
def predict(example, rules):

    for rule in rules:
        if rule_covers(example, rule):
            return "Yes"

    return "No"


df["Predicted"] = [
    predict(ex, foil_rules) for ex in examples
]

df["Correct"] = df["PlayTennis"] == df["Predicted"]

print("FOIL Prediction Results:")
display(df)

accuracy = df["Correct"].mean() * 100

print("\nModel Accuracy:", round(accuracy, 2), "%")
print("Correct Predictions:", df["Correct"].sum())
print("Incorrect Predictions:", (~df["Correct"]).sum())

FOIL Prediction Results:


,Outlook,Temperature,Humidity,Wind,PlayTennis,Predicted,Correct
0,Sunny,Hot,High,Weak,No,No,True
1,Sunny,Hot,High,Strong,No,No,True
2,Overcast,Hot,High,Weak,Yes,Yes,True
3,Rain,Mild,High,Weak,Yes,Yes,True
4,Rain,Cool,Normal,Weak,Yes,Yes,True
5,Rain,Cool,Normal,Strong,No,No,True
6,Overcast,Cool,Normal,Strong,Yes,Yes,True
7,Sunny,Mild,High,Weak,No,No,True
8,Sunny,Cool,Normal,Weak,Yes,Yes,True
9,Rain,Mild,Normal,Weak,Yes,Yes,True



Model Accuracy: 100.0 %
Correct Predictions: 14
Incorrect Predictions: 0


In [12]:
print("FINAL FOIL CLASSIFICATION RULES")
print("=" * 45)

for i, rule in enumerate(foil_rules, 1):
    conditions = [
        f"{attribute} = {value}"
        for attribute, value in rule
    ]

    print(f"\nRule {i}:")
    print("IF", " AND ".join(conditions))
    print("THEN PlayTennis = Yes")

print("\nDefault prediction: PlayTennis = No")

FINAL FOIL CLASSIFICATION RULES

Rule 1:
IF Outlook = Overcast
THEN PlayTennis = Yes

Rule 2:
IF Humidity = Normal AND Wind = Weak
THEN PlayTennis = Yes

Rule 3:
IF Temperature = Mild AND Humidity = Normal
THEN PlayTennis = Yes

Rule 4:
IF Outlook = Rain AND Wind = Weak
THEN PlayTennis = Yes

Default prediction: PlayTennis = No


In [13]:
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("=" * 45)

print("Algorithm: FOIL (First Order Inductive Learner)")
print("Dataset: Play Tennis")
print("Total examples:", len(df))
print("Attributes:", len(X.columns))
print("Rules generated:", len(foil_rules))
print("Correct predictions:", df["Correct"].sum())
print("Incorrect predictions:", (~df["Correct"]).sum())
print("Training accuracy:", round(accuracy, 2), "%")

EXPERIMENT COMPLETED SUCCESSFULLY
Algorithm: FOIL (First Order Inductive Learner)
Dataset: Play Tennis
Total examples: 14
Attributes: 4
Rules generated: 4
Correct predictions: 14
Incorrect predictions: 0
Training accuracy: 100.0 %
